In [ ]:
%matplotlib ipympl

from helper import *
import scipy.linalg as sci_lin

## LQR

In [ ]:
alpha = np.eye(3) * -1
A = np.array([[0, 1, 0], [0, 0, 1], [0, 0, 0]], dtype=float)
B = np.array([[0], [0], [1]], dtype=float)
Q = np.diag([1e0, 1e0, 1e0])
R = np.array([[1e-0]], dtype=float)
K, _, _ = ct.lqr(A - alpha, B, Q, R)
E = sci_lin.expm((A - B @ K) * spec.dt)

In [ ]:
@functools.partial(jax.jit, static_argnames=["n"])
def propogate_pos(x0, E, n):
    def scan_body(x0, _):
        x1 = E @ x0
        return x1, x1[0]

    _, res = jax.lax.scan(scan_body, x0, length=n)
    return res

In [ ]:
x0 = np.array([-1.0, -1.0, -1.0])
y = propogate_pos(x0, E, 200)
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(y)
ax.grid()

## quantitative analysis

In [ ]:
lti_int, x_data, y_data, ctrl_data = get_data(0)

In [ ]:
def pred_fun(hist, t, _):
    assert hist.shape == (3,)
    v0 = hist[-1]
    v1 = (hist[-1] - hist[-2]) / spec.dt
    v2 = jnp.squeeze(jnp.diff(jnp.diff(hist))) / spec.dt**2
    return propogate_pos(jnp.array([v0, v1, v2]), E, t.size)

pred_err(pred_fun, y_data, 3)

In [ ]:
def pred_fun_check(hist, t, idx):
    x0 = x_data[idx]
    _, y = lti_int(x0=x0, u=jnp.ones_like(t) * ctrl_data[idx])
    return y

pred_err(pred_fun_check, y_data, 2)

## visualize

In [ ]:
idx = 4444
t = np.linspace(0, 2.0, num=spec.n + 1, endpoint=True)

fig, axs = plt.subplots(2, 1, figsize=(7, 8))
axs[0].plot(y_data[idx + 1: idx + spec.n + 2], label="y_data")
axs[0].plot(pred_fun(y_data[idx - 2: idx + 1], t, idx), label="pred")
axs[1].plot(y_data[idx + 1: idx + spec.n + 2], label="y_data")
axs[1].plot(pred_fun_check(y_data[idx - 1: idx + 1], t, idx), label="flat_pred")
for ax in axs:
    ax.grid()
    ax.legend()